# Lagrangian Coherent Structure (LCS) Map Around the Cabo Verde Region

In [ ]:
import math
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
from scipy.interpolate import griddata

import parcels
from parcels import StatusCode

## Parameters

In [ ]:
lon_min = -29
lon_max = -22.5
lat_min = 14
lat_max = 19.5

refine_lon_fac = 5
refine_lat_fac = 5
# delta_m, deg_lat_m = 2000.0, 111320.0

# grid_points_lon = 200
# grid_points_lat = 200

reference_time = "2025-08-22"

integration_days = 10
integration_direction = -1

In [ ]:
flowfield_env = xr.open_dataset("ssc.nc")
flowfield_env

## Boundary Conditions

In [ ]:
# TODO: Let's implement with xr.where to be independent of numpy api
flowfield_env["uo"][:, :,  0] = 0.0
flowfield_env["uo"][:, :, -1] = 0.0
flowfield_env["uo"][:,  0, :] = 0.0
flowfield_env["uo"][:, -1, :] = 0.0
flowfield_env["vo"][:, :,  0] = 0.0
flowfield_env["vo"][:, :, -1] = 0.0
flowfield_env["vo"][:,  0, :] = 0.0
flowfield_env["vo"][:, -1, :] = 0.0

In [ ]:
ocean_mask = flowfield_env.uo.mean("time").notnull().astype(float)

In [ ]:
v_repel = 0.02

mask = ocean_mask.values
nj, ni = mask.shape
u_rep_arr = np.zeros((nj, ni))
v_rep_arr = np.zeros((nj, ni))

for j in range(1, nj - 1):
    for i in range(1, ni - 1):
        if mask[j, i] == 0:
            continue
        if mask[j,   i-1] == 0: u_rep_arr[j, i] += v_repel
        if mask[j,   i+1] == 0: u_rep_arr[j, i] -= v_repel
        if mask[j-1, i  ] == 0: v_rep_arr[j, i] += v_repel
        if mask[j+1, i  ] == 0: v_rep_arr[j, i] -= v_repel

In [ ]:
u_repel = xr.DataArray(u_rep_arr, coords=ocean_mask.coords, dims=ocean_mask.dims)
v_repel_field = xr.DataArray(v_rep_arr, coords=ocean_mask.coords, dims=ocean_mask.dims)

In [ ]:
flowfield_env["uo"] = flowfield_env.uo + u_repel
flowfield_env["vo"] = flowfield_env.vo + v_repel_field

In [ ]:
flowfield_env.uo.mean("time").plot()

## Parcels Fieldset

In [ ]:
fieldset = parcels.FieldSet.from_xarray_dataset(
    flowfield_env,
    variables={"U": "uo", "V": "vo"},
    dimensions={"lon": "longitude", "lat": "latitude", "time": "time"},
)

In [ ]:
fieldset.computeTimeChunk()

plt.pcolormesh(fieldset.U.grid.lon, fieldset.U.grid.lat, fieldset.U.data[11, :, :])
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.colorbar(label="Zonal velocity [m/s]")
plt.show()

## Setting up Particle Triplets

In [ ]:
particle_lon = xr.DataArray(
    data=np.linspace(
        flowfield_env.longitude.min().item(),
        flowfield_env.longitude.max().item(),
        refine_lon_fac * flowfield_env.sizes["longitude"] - 1
    ),
    name="plon",
    dims="plon",
)
particle_lat = xr.DataArray(
    data=np.linspace(
        flowfield_env.latitude.min().item(),
        flowfield_env.latitude.max().item(),
        refine_lat_fac * flowfield_env.sizes["latitude"] - 1
    ),
    name="plat",
    dims="plat",
)
particle_lon, particle_lat = xr.broadcast(particle_lon, particle_lat)

In [ ]:
eigenvector_plot_count = particle_lon.data.size
eigenvector_plot_count

In [ ]:
pset = parcels.ParticleSet.from_list(
    fieldset=fieldset,
    pclass=parcels.JITParticle,
    lon=particle_lon.stack(pid=["plon", "plat"]).load().data,
    lat=particle_lat.stack(pid=["plon", "plat"]).load().data,
    time=np.datetime64(reference_time),
)
print(pset)

In [ ]:
# plt.pcolormesh(fieldset.U.grid.lon, fieldset.U.grid.lat, fieldset.U.data[11, :, :])
# plt.xlabel("longitude")
# plt.ylabel("latitude")
# plt.colorbar()

fig, ax = plt.subplots(1, 1, figsize=(20, 20))
ax.plot(pset.lon, pset.lat, "ko", markersize=1)
plt.show()

## Particle Advection

In [ ]:
output_file = pset.ParticleFile(
    name="../data/CV_gridParticles.zarr",
    outputdt=timedelta(hours=1), 
)

In [ ]:
def CheckOutOfBounds(particle, fieldset, time):
    if particle.state == StatusCode.ErrorOutOfBounds:
        particle.delete()

def CheckError(particle, fieldset, time):
    if particle.state >= 50:
        particle.delete()

kernels = (
    pset.Kernel(parcels.AdvectionRK4)
    + pset.Kernel(CheckOutOfBounds)
)

pset.execute(
    kernels,
    runtime=timedelta(days=integration_days),
    dt=timedelta(minutes=int(integration_direction * 5)),
    #output_file=output_file,
)

In [ ]:
print(pset)

In [ ]:
# plt.pcolormesh(fieldset.U.grid.lon, fieldset.U.grid.lat, fieldset.U.data[11, :, :])
# plt.xlabel("longitude")
# plt.ylabel("latitude")
# plt.colorbar()

fig, ax = plt.subplots(1, 1, figsize=(20, 20))
ax.plot(pset.lon, pset.lat, "ko", markersize=1)
plt.show()

## Deformation Gradient Tensor

In [ ]:
final_lon = (particle_lon.stack(pid=["plon", "plat"]) * 0 + pset.lon).unstack()
final_lat = (particle_lat.stack(pid=["plon", "plat"]) * 0 + pset.lat).unstack()

In [ ]:
final_lon

In [ ]:
dX = (final_lon.shift(plon=-1) - final_lon.shift(plon=1)) * 111e3 * np.cos(np.deg2rad(final_lat))
dx = (particle_lon.shift(plon=-1) - particle_lon.shift(plon=1)) * 111e3 * np.cos(np.deg2rad(particle_lat))

dY = (final_lat.shift(plat=-1) - final_lat.shift(plat=1)) * 111e3
dy = (particle_lat.shift(plat=-1) - particle_lat.shift(plat=1)) * 111e3

In [ ]:
F = np.array([
    [(dX/dx).stack(pid=["plon", "plat"]), (dX/dy).stack(pid=["plon", "plat"])],
    [(dY/dx).stack(pid=["plon", "plat"]), (dY/dy).stack(pid=["plon", "plat"])],
])

In [ ]:
F.shape

## Cauchy-Green Strain Tensor computation

In [ ]:
C   = np.einsum("kil,kjl->ijl", F, F) 
print(C.shape)
C_T = np.transpose(C, (2, 0, 1))      
eigenvalues, eigenvectors = np.linalg.eigh(C_T)

In [ ]:
ev_max = eigenvectors[:, :, -1]

In [ ]:
eigenvalues.shape

In [ ]:
eval_max = eigenvalues[:, -1]

In [ ]:
eval_max.shape

## LCS maps

In [ ]:
T = integration_days * 24 * 60 * 60

In [ ]:
max_eigenvalues = (np.max(eigenvalues, axis=1) + particle_lon.stack(pid=["plon", "plat"]) * 0).rename("lambda_plus").unstack()
max_eigenvalues = max_eigenvalues.assign_coords(
    plon=particle_lon.isel(plat=0, drop=True),
    plat=particle_lat.isel(plon=0, drop=True),
)
max_eigenvalues

In [ ]:
max_eigenvalues.drop_encoding().to_netcdf(f"010_max_eigenvalues_{reference_time}.nc")

In [ ]:
max_eigenvalues.plot(x="plon", y="plat", size=20)

In [ ]:
FTLE = (1/(2*T)) * np.log(np.maximum(max_eigenvalues, 1e-10))
FTLE

In [ ]:
FTLE.drop_encoding().to_netcdf(f"010_FTLE_{reference_time}.nc")

In [ ]:
FTLE.plot(x="plon", y="plat", size=20)

In [ ]:
#FTLE_da = xr.DataArray(FTLE, 
#                      dims = ("pos",),
#                      coords = {"lon":(("pos",), lons[0::3]),
#                                "lat":(("pos",), lats[0::3])},
#                       name = "FTLE"
#                      )

In [ ]:
#FTLE_da

In [ ]:
#FTLE_da.to_dataset().to_netcdf("010_FTLE.nc")

### FTLE fields regrid

In [ ]:
hess_eigvals, hess_eigvecs = np.linalg.eigh(H)

In [ ]:
lambda_min = hess_eigvals[..., 0]
e_min = hess_eigvecs[..., :, 0]  

In [ ]:
grad_dot_emin = phi_x * e_min[..., 0] + phi_y * e_min[..., 1]

In [ ]:
min_ridge_strength = np.nanpercentile(lambda_min, 10)

In [ ]:
ridge_candidate = lambda_min < min_ridge_strength
g_masked = np.where(ridge_candidate, grad_dot_emin, np.nan)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
pc = ax.pcolormesh(LON, LAT, FTLE_grid, shading="auto", cmap="viridis")
plt.colorbar(pc, ax=ax, label="FTLE (1/s)")

ridge_lines = ax.contour(LON, LAT, g_masked, levels=[0], colors="black", linewidths=2)

ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
plt.show()

In [ ]:
ev_min_particles = eigenvectors[:, :, 0]

theta_p = np.arctan2(ev_min_particles[:, 1], ev_min_particles[:, 0])
cos2_p, sin2_p = np.cos(2 * theta_p), np.sin(2 * theta_p)

cos2_grid = griddata(np.column_stack([center_lon, center_lat]), cos2_p, (LON, LAT), method="cubic")
sin2_grid = griddata(np.column_stack([center_lon, center_lat]), sin2_p, (LON, LAT), method="cubic")
theta_grid = 0.5 * np.arctan2(sin2_grid, cos2_grid)

In [ ]:
evmin_x, evmin_y = np.cos(theta_grid), np.sin(theta_grid)
tangent_x, tangent_y = -e_min[..., 1], e_min[..., 0]
cos_align = np.abs(tangent_x * evmin_x + tangent_y * evmin_y)
alignment_deg = np.degrees(np.arccos(np.clip(cos_align, 0, 1)))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
pc = ax.pcolormesh(LON, LAT, alignment_deg, shading="auto", cmap="RdBu_r", vmin=0, vmax=90)
plt.colorbar(pc, ax=ax,)
ax.contour(LON, LAT, g_masked, levels=[0], colors="black", linewidths=3)

ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
plt.show()